# Sup Figure 1 | Strand-Bias and Insert-Level Fitness

## Configuration

In [ ]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import numpy as np


In [ ]:
# --- data directory (populate yourself -- see README's Data section) ---
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- paths ---
GENE_DATA_PATH = DATA_DIR / "gene_fitness_results_with_annotations.parquet"
DIRECTIONAL_LM_PATH = DATA_DIR / "directional_lmer_stats.parquet"

# --- analysis parameters ---
CONDITION = "LB_4_salt"


In [ ]:
df_raw = pd.read_parquet(GENE_DATA_PATH)
print(f"{len(df_raw):,} rows loaded")
salt_genes = df_raw[df_raw["condition"] == CONDITION].copy()
print(f"{len(salt_genes):,} rows filtered")


In [ ]:
pd.set_option('display.max_columns', None)
salt_gene_hits = salt_genes[salt_genes["gene_call"] == "Hit"]
salt_gene_hits.head()


In [ ]:
salt_direction_stats = pd.read_parquet(DIRECTIONAL_LM_PATH)

In [ ]:
salt_direction_stats.sort_values(by='lmer_rev_p_value', ascending=True, inplace=True)
salt_direction_stats.rename(columns={"lmer_estimate": "lmer_sense_estimate"}, inplace=True)
salt_direction_stats["fract_sense_inserts"] = salt_direction_stats["lmer_n_sense_inserts"] / (salt_direction_stats["lmer_n_sense_inserts"] + salt_direction_stats["lmer_n_antisense_inserts"])

In [ ]:
salt_hits_extended = salt_gene_hits.merge(salt_direction_stats[["locus_tag", "lmer_sense_estimate", "lmer_antisense_estimate", "lmer_rev_estimate", "lmer_rev_p_value", "fract_sense_inserts"]], 
                                       on = "locus_tag", how = "left")

salt_hits_extended

In [ ]:
salt_hits_extended_filt = salt_hits_extended[(salt_hits_extended["n_sense_inserts"] >= 2) & 
                                            (salt_hits_extended["n_antisense_inserts"] >= 2)].copy()

salt_hits_extended_filt.sort_values(by='lmer_rev_p_value', ascending=False, inplace=True)
salt_hits_extended_filt["p_value_fdr"] = multipletests(salt_hits_extended_filt["lmer_rev_p_value"], method="fdr_bh")[1]
salt_hits_extended_filt["neg_log10_p_value_fdr"] = -np.log10(salt_hits_extended_filt["p_value_fdr"])
salt_hits_extended_filt["sig_p_value_fdr"] = salt_hits_extended_filt["p_value_fdr"] < 0.05
salt_hits_extended_filt["sig_p_value_fdr"].value_counts()

### Sense vs. antisense strand fitness estimates Supplementary 1A

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(salt_hits_extended_filt, x="lmer_sense_estimate", y="lmer_antisense_estimate", hue = "sig_p_value_fdr", s=10, ax=ax)
plt.axline((0,0), slope = 1, color="black", linewidth=0.5)
plt.xlabel("Sense Strand Estimate")
plt.ylabel("Antisense Strand Estimate")
plt.legend(title = "Significant\nStrand Effect", loc = "upper left")

fig.savefig(RESULTS_DIR / "figureS1a_sense_antisense_scatter.pdf", bbox_inches="tight")

plt.show()


### Forward-strand effect size distribution Supplementary 1B

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

salt_hits_extended_filt["lmer_fwd_estimate"] = salt_hits_extended_filt["lmer_sense_estimate"] - salt_hits_extended_filt["lmer_antisense_estimate"]
sns.histplot(salt_hits_extended_filt[~salt_hits_extended_filt["lmer_fwd_estimate"].isna()], x="lmer_fwd_estimate",
             bins=20, ax=ax)

median_effect_size = salt_hits_extended_filt["lmer_fwd_estimate"].median()

plt.ylabel("Number of Genes")
plt.xlabel("Gene Sense Estimate - Antisense Estimate")
plt.axvline(median_effect_size, color="black", linewidth=1, linestyle="--")

fig.savefig(RESULTS_DIR / "figureS1b_hist_sense_antisense_estimate_diff.pdf", bbox_inches="tight")

plt.show()

print(median_effect_size)

### Insert level plots Supplementary 1C

In [ ]:
INSERT_DATA_PATH = DATA_DIR / "selection_experiment_insert_data.parquet"
inserts = pd.read_parquet(INSERT_DATA_PATH)
print(f"{len(inserts):,} rows loaded")


In [ ]:
# --- Step 1: mean empty-insert fitness per replicate per environment ---
empty_bc_fitness = (
    inserts[inserts["insert_type"] == "empty_insert"]
    .groupby(["environment", "replicate"])
    .agg(
        mean_empty_bc_fitness=("fitness", "mean"),
        n_empty_bcs=("bc_sequence", "nunique"),
    )
    .reset_index()
)

# --- Step 2: center each insert against its replicate empty mean,
#             then average across replicates per barcode ---
inserts_ext = inserts.merge(empty_bc_fitness, on=["environment", "replicate"], how="left")
inserts_ext["fitness_centered"] = inserts_ext["fitness"] - inserts_ext["mean_empty_bc_fitness"]

bc_fitness_means = (
    inserts_ext[inserts_ext["insert_type"] != "unknown"]
    .groupby(["environment", "insert_type", "bc_sequence"])
    .agg(
        mean_centered_fitness=("fitness_centered", "mean"),
        std_fitness=("fitness_centered", "std"),
    )
    .reset_index()
)

# --- Step 3: re-correct against the empty-insert barcode distribution per environment ---
empty_stats = (
    bc_fitness_means[bc_fitness_means["insert_type"] == "empty_insert"]
    .groupby("environment")
    .agg(
        mean_empty_bc_fitness=("mean_centered_fitness", "mean"),
        std_empty_bc_fitness=("mean_centered_fitness", "std"),
    )
    .reset_index()
)
bc_fitness_means = bc_fitness_means.merge(empty_stats, on="environment", how="left")
bc_fitness_means["mean_fitness_corrected"] = (
    bc_fitness_means["mean_centered_fitness"] - bc_fitness_means["mean_empty_bc_fitness"]
)
bc_fitness_means["z_score"] = (
    (bc_fitness_means["mean_centered_fitness"] - bc_fitness_means["mean_empty_bc_fitness"])
    / bc_fitness_means["std_empty_bc_fitness"]
)

print(bc_fitness_means.groupby(["environment", "insert_type"]).size())

# Build one row per barcode carrying positional columns + mean_fitness_corrected.
# The raw inserts table has one row per barcode per replicate; groupby.first()
# collapses replicates so each barcode appears exactly once.
corrected = bc_fitness_means[
    bc_fitness_means["environment"] == CONDITION
][["bc_sequence", "mean_fitness_corrected"]]

annotated = (
    inserts[
        (inserts["environment"] == CONDITION)
    ]
    .merge(corrected, on="bc_sequence", how="inner")
    .groupby("bc_sequence")
    .first()
    .reset_index()
)

print(f"{len(annotated):,} unique {CONDITION} barcodes with corrected fitness")

In [ ]:
annotated_sense_long = annotated[["bc_sequence", "mean_fitness_corrected", "sense_gene_ids"]].explode("sense_gene_ids").dropna(subset=["sense_gene_ids"]).rename(columns={"sense_gene_ids": "gene_ids"})
annotated_sense_long["direction"] = "sense"
annotated_antisense_long = annotated[["bc_sequence", "mean_fitness_corrected", "antisense_gene_ids"]].explode("antisense_gene_ids").dropna(subset=["antisense_gene_ids"]).rename(columns={"antisense_gene_ids": "gene_ids"})
annotated_antisense_long["direction"] = "antisense"
annotated_long = pd.concat([annotated_sense_long, annotated_antisense_long])
annotated_long

In [ ]:
genes_of_interest = [ "gene-b1837", "gene-PYW03_RS06940", "gene-b2516",]
annotated_long_filt = annotated_long[annotated_long["gene_ids"].isin(genes_of_interest)]

fig, ax = plt.subplots(figsize=(4, 4))

### Use two color pallete of gold and purple
sns.boxplot(annotated_long_filt, y="mean_fitness_corrected", x="gene_ids", hue = "direction", palette = ["#FFD700", "#800080"], dodge=True, fliersize=0,
            order = ["gene-b1837", "gene-PYW03_RS06940", "gene-b2516"])
sns.stripplot(annotated_long_filt, y="mean_fitness_corrected", x="gene_ids", s=3, hue = "direction", palette = ["black", "black"], dodge=True, legend=False,
              order = ["gene-b1837", "gene-PYW03_RS06940", "gene-b2516"])

### Manually change x tick labels
plt.xticks([0, 1, 2], ["E. coli yebW", "P. putida mraY", "E. coli rodZ"], ha="right", rotation=45)
plt.xlabel("")
plt.ylabel("Mean Insert Fitness")
plt.legend(title = "Direction", loc = "upper right")

fig.savefig(RESULTS_DIR / "figureS1c_insert_level_boxplot.pdf", bbox_inches="tight")
plt.show()